# ✏️ Doodle-to-Emoji — train your own model

**In Google Colab, select Runtime → Run all once.** This notebook installs its
dependencies, streams a small Quick Draw training set, trains your three-class
CNN, converts and loads it, then opens the Gradio drawing app. The final cell
automatically enables a temporary public Gradio link in Colab. Keep the runtime
connected while using the app. A CPU runtime is sufficient; a GPU is optional.

The model learns **apple 🍎, star ⭐, and sun ☀️** from scratch. It replaces the
earlier pretrained classifier. There is no pretrained download or manual
reload step. The browser's Gradio Sketchpad provides the drawing canvas;
Python handles preprocessing and prediction on the notebook runtime.

The first run needs internet access and takes a few minutes for installation,
data loading, and training. Later Run all passes reuse the cached training
images and train a new model. Colab's temporary files disappear when its runtime
is deleted, so download the notebook/model if you want to keep them.


## 1. Install dependencies

Run this before the imports. TensorFlow trains and exports the CNN; LiteRT loads
the exported model; Gradio provides the web interface. These versions have
Python 3.12 and 3.13 Linux wheels for Colab.


In [ ]:
%pip install -q "tensorflow==2.20.0" "gradio==5.49.1" "ai-edge-litert==2.2.0" "numpy>=1.26,<2.3" "Pillow>=10,<13"


## 2. Imports and settings

The defaults need no edits. To add a category later, add its Quick Draw name and
an emoji to `CATEGORY_EMOJIS`, then Run all again. Label order is saved with the
model, so the app uses exactly the vocabulary it was trained on.


In [ ]:
import hashlib
import html
import json
import os
import sys
import threading
import time
import urllib.request
from pathlib import Path
from urllib.parse import quote

os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageOps
from ai_edge_litert.interpreter import Interpreter

CATEGORY_EMOJIS = {"apple": "🍎", "star": "⭐", "sun": "☀️"}
CATEGORIES = list(CATEGORY_EMOJIS)
SAMPLES_PER_CATEGORY = 2000
EPOCHS = 5
BATCH_SIZE = 32
SEED = 42
CANVAS_SIZE = 384

assert len(CATEGORIES) >= 2
assert SAMPLES_PER_CATEGORY >= 20
tf.keras.utils.set_random_seed(SEED)

# Keep custom weights separate from the earlier pretrained model cache.
MODEL_DIR = Path("work/custom_doodle_model")
DATA_DIR = Path("work/quickdraw_training")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / "model.tflite"
labels_path = MODEL_DIR / "labels.txt"
print(f"Training {len(CATEGORIES)} classes: {', '.join(CATEGORIES)}")


## 3. Prepare drawings and load a small training set

Training sketches and user drawings use the **same** crop, centering, polarity,
and 28 × 28 normalization. Raw Quick Draw coordinates are scaled to fit the
canvas before rasterization, so large sketches are not clipped.

Only the first 2,000 usable recognized sketches per category are streamed from
the [official Quick Draw dataset](https://github.com/googlecreativelab/quickdraw-dataset).
The connection closes once enough examples have been read, avoiding full
hundreds-of-megabytes downloads. Processed images are cached for later runs.


In [ ]:
def preprocess_sketch(value):
    """Return float32 [1, 28, 28, 1] and a PIL preview, or reject a blank canvas."""
    if isinstance(value, dict):  # Gradio Sketchpad sends an EditorValue dictionary.
        value = value.get("composite")
    if value is None:
        raise ValueError("Draw something first.")
    image = value if isinstance(value, Image.Image) else Image.fromarray(value)
    # Transparent pixels are empty paper; dropping alpha would turn them black.
    rgba = image.convert("RGBA")
    paper = Image.new("RGBA", rgba.size, "white")
    gray = Image.alpha_composite(paper, rgba).convert("L")
    pixels = np.asarray(gray)
    ink = pixels < 240
    ys, xs = np.where(ink)
    if len(xs) < 20 or max(int(np.ptp(xs)), int(np.ptp(ys))) < 8:
        raise ValueError("The canvas is empty or too small. Draw a larger sketch.")
    crop = gray.crop((int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1))
    # Training and inference both use bright strokes on a dark background.
    crop = ImageOps.invert(crop)
    scale = 26 / max(crop.size)
    size = tuple(max(1, round(dimension * scale)) for dimension in crop.size)
    crop = crop.resize(size, Image.Resampling.LANCZOS)
    preview = Image.new("L", (28, 28), 0)
    preview.paste(crop, ((28 - size[0]) // 2, (28 - size[1]) // 2))
    tensor = np.asarray(preview, dtype=np.float32)[None, :, :, None] / 255.0
    return tensor, preview


In [ ]:
def rasterize_training_drawing(drawing, line_width=8):
    strokes = [list(zip(stroke[0], stroke[1])) for stroke in drawing if len(stroke) >= 2]
    points = [point for stroke in strokes for point in stroke]
    if not points:
        raise ValueError("Empty training drawing")
    coordinates = np.asarray(points, dtype=np.float32)
    lower = coordinates.min(axis=0)
    extent = coordinates.max(axis=0) - lower
    span = float(extent.max())
    if span <= 0 or not np.all(np.isfinite(coordinates)):
        raise ValueError("Invalid training drawing")
    scale = (CANVAS_SIZE - 64) / span
    offset = (CANVAS_SIZE - extent * scale) / 2
    image = Image.new("RGB", (CANVAS_SIZE, CANVAS_SIZE), "white")
    pen = ImageDraw.Draw(image)
    for stroke in strokes:
        normalized = [tuple((np.asarray(point) - lower) * scale + offset) for point in stroke]
        if len(normalized) >= 2:
            pen.line(normalized, fill="black", width=line_width, joint="curve")
        elif normalized:
            x, y = normalized[0]
            radius = line_width / 2
            pen.ellipse((x - radius, y - radius, x + radius, y + radius), fill="black")
    _, preview = preprocess_sketch(image)
    return np.asarray(preview, dtype=np.uint8)


def load_training_data():
    # Changing classes, sample count, or preprocessing creates a new cache key.
    config = {"categories": CATEGORIES, "count": SAMPLES_PER_CATEGORY, "preprocessing": 1}
    key = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
    cache = DATA_DIR / f"images_{key}.npz"
    if cache.exists():
        with np.load(cache, allow_pickle=False) as saved:
            images, targets = saved["images"], saved["targets"]
        print(f"Loaded {len(targets):,} cached training images.")
    else:
        images, targets = [], []
        for class_index, category in enumerate(CATEGORIES):
            url = f"https://storage.googleapis.com/quickdraw_dataset/full/raw/{quote(category, safe='')}.ndjson"
            print(f"Streaming {category}...", flush=True)
            accepted = 0
            with urllib.request.urlopen(url, timeout=120) as response:
                for line in response:
                    record = json.loads(line)
                    if not record.get("recognized", False):
                        continue
                    try:
                        # A few stroke widths help match different browser brush sizes.
                        image = rasterize_training_drawing(record["drawing"], 6 + 2 * (accepted % 4))
                    except (ValueError, KeyError, IndexError):
                        continue
                    images.append(image)
                    targets.append(class_index)
                    accepted += 1
                    if accepted == SAMPLES_PER_CATEGORY:
                        break
            if accepted < SAMPLES_PER_CATEGORY:
                raise RuntimeError(f"Only found {accepted} usable {category} sketches.")
            print(f"  Prepared {accepted:,} {category} sketches.", flush=True)
        images = np.stack(images)
        targets = np.asarray(targets, dtype=np.int32)
        temporary = cache.with_suffix(".tmp.npz")
        np.savez_compressed(temporary, images=images, targets=targets)
        temporary.replace(cache)
    expected = SAMPLES_PER_CATEGORY * len(CATEGORIES)
    if images.shape != (expected, 28, 28) or targets.shape != (expected,):
        raise RuntimeError(f"Invalid dataset cache. Delete {cache} and Run all again.")
    return images[..., None].astype(np.float32) / 255.0, targets


X, y = load_training_data()
print(f"Dataset: {X.shape}, values {X.min():.1f}–{X.max():.1f}")


## 4. Train the custom CNN

Use 90% of each category for training and 10% for validation. The CNN is
initialized from scratch. The reported validation accuracy measures held-out
Quick Draw sketches; it is not a guarantee for every user's drawings.


In [ ]:
rng = np.random.default_rng(SEED)
train_indices, validation_indices = [], []
for class_index in range(len(CATEGORIES)):
    indices = rng.permutation(np.flatnonzero(y == class_index))
    validation_count = max(1, int(len(indices) * 0.1))
    validation_indices.extend(indices[:validation_count])
    train_indices.extend(indices[validation_count:])
train_indices = rng.permutation(train_indices)
validation_indices = np.asarray(validation_indices)
X_train, y_train = X[train_indices], y[train_indices]
X_val, y_val = X[validation_indices], y[validation_indices]

# Use a small data-loading thread pool so CPU-only notebook runtimes work too.
options = tf.data.Options()
options.threading.private_threadpool_size = 2
train_data = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(len(y_train), seed=SEED)
    .batch(BATCH_SIZE).with_options(options).prefetch(1)
)
validation_data = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE).with_options(options).prefetch(1)
)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(CATEGORIES), activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_data, validation_data=validation_data, epochs=EPOCHS, verbose=2)
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.1%}")


## 5. Export and immediately load the new model

Save the new model and its labels together, then create a **new interpreter**
from those files. The app below uses this interpreter. Nothing downloads or
restores the old pretrained model.


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()
model_path.write_bytes(tflite_bytes)
labels_path.write_text("\n".join(CATEGORIES) + "\n", encoding="utf-8")
(MODEL_DIR / "training_info.json").write_text(json.dumps({
    "categories": CATEGORIES,
    "samples_per_category": SAMPLES_PER_CATEGORY,
    "epochs": EPOCHS,
    "seed": SEED,
    "validation_accuracy": float(history.history["val_accuracy"][-1]),
}, indent=2), encoding="utf-8")
print(f"Saved custom model: {model_path} ({len(tflite_bytes) / 1024:.0f} KiB)")


In [ ]:
LABELS = labels_path.read_text(encoding="utf-8").splitlines()
assert LABELS == CATEGORIES, "The saved labels must match training order."
EMOJI_MAP = {label: CATEGORY_EMOJIS[label] for label in LABELS}
interpreter = Interpreter(model_path=str(model_path), num_threads=2)
interpreter.allocate_tensors()
INPUT = interpreter.get_input_details()[0]
OUTPUT = interpreter.get_output_details()[0]
MODEL_LOCK = threading.Lock()
assert tuple(INPUT["shape"]) == (1, 28, 28, 1)
assert tuple(OUTPUT["shape"]) == (1, len(LABELS))
print(f"App loaded the custom model with labels: {LABELS}")


## 6. Prediction callbacks and checks

Verify that the exported model's predictions agree with the trained TensorFlow
model. The example star then exercises the same callback as the web app.


In [ ]:
def model_probabilities(tensor):
    """Respect the TFLite input/output quantization scales and zero points."""
    dtype = INPUT["dtype"]
    if np.issubdtype(dtype, np.integer):
        scale, zero_point = INPUT["quantization"]
        limits = np.iinfo(dtype)
        model_input = np.clip(np.rint(tensor / scale + zero_point), limits.min, limits.max).astype(dtype)
    else:
        model_input = tensor.astype(dtype)
    # An Interpreter cannot process simultaneous calls safely.
    with MODEL_LOCK:
        interpreter.set_tensor(INPUT["index"], model_input)
        interpreter.invoke()
        output = interpreter.get_tensor(OUTPUT["index"])[0].copy()
    probabilities = output.astype(np.float32)
    if np.issubdtype(OUTPUT["dtype"], np.integer):
        scale, zero_point = OUTPUT["quantization"]
        probabilities = (probabilities - zero_point) * scale
    if not np.all(np.isfinite(probabilities)) or probabilities.sum() <= 0:
        raise RuntimeError("The model returned invalid scores.")
    return probabilities


def rank_emojis(probabilities, top_k=3):
    """Sum scores for categories sharing an emoji; keep unsupported mass unassigned."""
    grouped = {}
    for label, probability in zip(LABELS, probabilities):
        emoji = EMOJI_MAP.get(label.lower())
        score = float(probability)
        if emoji is None or score <= 0:
            continue
        if emoji not in grouped:
            grouped[emoji] = {"emoji": emoji, "label": label, "score": 0.0, "strongest": -1.0}
        item = grouped[emoji]
        item["score"] += score
        if score > item["strongest"]:
            item.update(label=label, strongest=score)
    # Do not renormalize over mapped labels: that would inflate weak matches.
    return sorted(grouped.values(), key=lambda item: item["score"], reverse=True)[:top_k]


In [ ]:
READY_HTML = '<div class="emoji-result">✏️</div><p class="result-caption">Your emoji appears here</p>'


def predict_emoji(value):
    start = time.perf_counter()
    try:
        tensor, preview = preprocess_sketch(value)
    except ValueError as exc:
        return READY_HTML, [], str(exc), None, ""
    probabilities = model_probabilities(tensor)
    matches = rank_emojis(probabilities)
    raw_index = int(probabilities.argmax())
    raw_label = LABELS[raw_index]
    raw_score = float(probabilities[raw_index])
    if not matches:
        return READY_HTML, [], f"I see '{raw_label}', but it has no supported emoji match.", preview, ""
    best = matches[0]
    heading = "Closest supported emoji" if best["score"] >= 0.60 else "Tentative emoji suggestion"
    result = (
        f'<div class="emoji-result">{html.escape(best["emoji"])}</div>'
        f'<p class="result-caption">{heading}: {html.escape(best["label"])}</p>'
    )
    rows = [[item["emoji"], item["label"], f'{item["score"]:.1%}'] for item in matches]
    status = f"Leading sketch category: {raw_label} ({raw_score:.1%})."
    if raw_label.lower() not in EMOJI_MAP:
        status += " That category has no emoji mapping; the suggestions are alternatives."
    if best["score"] < 0.60:
        status += " Try a larger, simpler outline."
    status += f" Prediction: {(time.perf_counter() - start) * 1000:.0f} ms."
    return result, rows, status, preview, best["emoji"]


def blank_canvas():
    return Image.new("RGB", (CANVAS_SIZE, CANVAS_SIZE), "white")


def reset_canvas():
    # None resets the entire editor, including brush strokes and undo history.
    return None, READY_HTML, [], "Draw one object, then click Match emoji.", None, ""


def make_example_star():
    """A locally generated sketch for checking the complete pipeline."""
    image = blank_canvas()
    points = []
    for index in range(10):
        angle = -np.pi / 2 + index * np.pi / 5
        radius = 140 if index % 2 == 0 else 58
        points.append((192 + radius * np.cos(angle), 192 + radius * np.sin(angle)))
    ImageDraw.Draw(image).line(points + [points[0]], fill="black", width=12, joint="curve")
    return image


In [ ]:
# Check a few held-out images per class, rather than assuming any sketch is correct.
check_indices = np.concatenate([np.flatnonzero(y_val == i)[:3] for i in range(len(LABELS))])
check_images = X_val[check_indices]
keras_scores = model(check_images, training=False).numpy()
lite_scores = np.stack([model_probabilities(image[None, ...]) for image in check_images])
np.testing.assert_allclose(lite_scores, keras_scores, rtol=1e-4, atol=1e-5)
np.testing.assert_allclose(lite_scores.sum(axis=1), 1.0, atol=1e-5)
assert rank_emojis(lite_scores[0])[0]["label"] == LABELS[int(lite_scores[0].argmax())]
assert predict_emoji(blank_canvas())[1] == []
assert reset_canvas()[0] is None
print(f"Passed: TensorFlow/LiteRT agreement on {len(check_images)} held-out sketches, label mapping, and blank-canvas handling.")

star_result = predict_emoji(make_example_star())
print(f"Example star: {star_result[4]} — {star_result[2]}")
from IPython.display import display
display(star_result[3])


## 7. Build the Gradio drawing app

Draw with your mouse, trackpad, or touchscreen. **Clear** resets the canvas,
undo history, predictions, and preview. The displayed categories come from
the new model's labels.


In [ ]:
CSS = """
.emoji-result {font-size: 112px; text-align: center; line-height: 1.5;}
.result-caption {text-align: center; font-size: 18px;}
"""


def build_app():
    with gr.Blocks(title="Doodle-to-Emoji", theme=gr.themes.Soft(), css=CSS, analytics_enabled=False) as demo:
        gr.Markdown("# ✏️ Doodle-to-Emoji\nDraw one object and find its closest supported emoji.")
        gr.Markdown("Try **" + ", ".join(LABELS) + "**. Use a simple outline.")
        with gr.Row():
            with gr.Column():
                canvas = gr.Sketchpad(
                    value=blank_canvas(), type="pil", image_mode="RGBA",
                    canvas_size=(CANVAS_SIZE, CANVAS_SIZE), fixed_canvas=True,
                    height=450, label="Draw here", layers=False, transforms=(),
                    brush=gr.Brush(colors=["#000000"], color_mode="fixed", default_size=6),
                    format="png",
                )
                with gr.Row():
                    match_button = gr.Button("Match emoji", variant="primary")
                    # Clear locally so canvas reset does not wait for a Colab round trip.
                    clear_button = gr.ClearButton(canvas, value="Clear")
                    example_button = gr.Button("Try a star")
            with gr.Column():
                result = gr.HTML(READY_HTML)
                suggestions = gr.Dataframe(
                    headers=["Emoji", "Sketch category", "Model score"],
                    datatype=["str", "str", "str"], value=[], interactive=False,
                    label="Up to three suggestions",
                )
                emoji_text = gr.Textbox(label="Copy your emoji", show_copy_button=True, interactive=False)
                status = gr.Textbox(value="Draw one object, then click Match emoji.", label="Status", interactive=False)
                with gr.Accordion("What the model sees", open=False):
                    preview = gr.Image(label="28 × 28 model input", type="pil", image_mode="L", height=180)
                gr.Markdown(
                    "Scores come from sketch classification; they are not calibrated accuracy. "
                    "This demo matches a fixed emoji vocabulary, so unsupported drawings can be misclassified."
                )
        outputs = [result, suggestions, status, preview, emoji_text]
        match_button.click(predict_emoji, inputs=canvas, outputs=outputs, concurrency_limit=1)
        # Reset to None first, then restore white paper. Repeating the same image
        # (or None) alone can leave the editor's internal strokes unchanged.
        clear_button.click(
            reset_canvas, inputs=[], outputs=[canvas] + outputs, queue=False,
        ).then(blank_canvas, inputs=[], outputs=canvas, queue=False)
        canvas.clear(lambda: reset_canvas()[1:], inputs=[], outputs=outputs, queue=False)
        example_button.click(make_example_star, inputs=[], outputs=canvas).then(
            predict_emoji, inputs=canvas, outputs=outputs, concurrency_limit=1,
        )
        with gr.Accordion("Supported sketch categories", open=False):
            gr.Markdown(", ".join(f"{emoji} {label}" for label, emoji in sorted(EMOJI_MAP.items())))
    return demo


## 8. Launch — this is the last code cell

In Colab, the public Gradio URL is enabled automatically and is printed below.
Open it if the inline app does not appear. The link works while the Colab
runtime remains active. Locally, the app uses a localhost URL instead.

**Try a star → Clear → draw again → Match emoji.** No earlier cell needs to be
rerun to activate the trained model.


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if "demo" in globals():
    demo.close()
demo = build_app()
demo.launch(inline=True, share=IN_COLAB, show_error=True, prevent_thread_lock=True)


## Save your work / hackathon notes

- **Stack:** Python + TensorFlow/Keras training + LiteRT inference + Pillow/NumPy
  image preparation + Gradio Blocks/Sketchpad interface. Colab runs the Python
  server; Gradio supplies the browser canvas and temporary sharing link.
- **Vocabulary:** only the categories listed in `CATEGORY_EMOJIS`. This is a
  sketch classifier followed by a category-to-emoji lookup, so it can choose
  one of those classes even when a drawing is unsupported.
- **Model files:** `work/custom_doodle_model/model.tflite`, `labels.txt`, and
  `training_info.json`. In Colab, use the Files sidebar to download these before
  ending the runtime. Download the notebook through File → Download → .ipynb.
- **Data:** Google's [Quick Draw dataset](https://github.com/googlecreativelab/quickdraw-dataset),
  shared under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
  This notebook selects recognized apple/star/sun sketches, rasterizes them,
  and normalizes them for training. Credit the dataset in your presentation.
